<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

he raw table's grain is one row = one content page, on one report date, for one client. For this contract, I aggregate that up to one row = one content page within March 2026.
Table(s): fact_content_daily_performance, partition month=2026-03 (plus dim_clients to check per-client history start dates).
Time window: the full calendar month of March 2026 — a mid-panel month, not the sealed final month (June 2026).

In [3]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Fact: row count + date span for this slice
summary = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_pages, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {DAILY}
""").df()
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_pages,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


## 2. Fields: feature / label / context / excluded

Feature (knowable before the decision moment): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, scroll_events — all trailing, already-logged daily metrics from before the moment I'd make a review decision.
Label/proxy: whether a page's traffic declines over time — the thing I'm ultimately trying to predict. Built from gsc_clicks/gsc_impressions trends, never itself a feature.
Context (grouping/joining only, never fed to a model): content_hash_id, client_hash_id, report_date, month.
Excluded: the ai_* breakdown columns (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other) — I'm leaving these out this week because they're a newer, sparser signal (AI-referral traffic) layered on top of the core search/analytics story, and mixing them in before I understand their coverage risks reading noise as pattern. I'll revisit them once the core features are solid.

In [4]:
con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 0").df()


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)
Grain: verified — zero rows where report_date × client × content repeats, confirming one row = one page-day-client.
Row count + span: 9,841,378 rows, 331,437 pages, 55 clients, full month 2026-03-01 to 2026-03-31 (from Step 1).
Availability: filtered ga4_data_available IS TRUE — see output above for what fraction of rows actually have usable GA4 data.
Five features: impressions_month, clicks_month, ctr_month, avg_position_month, days_active_month — each knowable at decision time because they're built only from March's already-logged daily rows, no future data touched.
The trap: added leaky_pct_change, a column computed directly from the same clicks that define declined_label. The score jumped from the honest baseline toward 1.0 — proof the model was reading the answer, not learning a pattern. Removed it and kept the honest score.

In [5]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# ---- 3a. Three proofs ----

# Proof 1: grain check (should return 0 rows)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {DAILY}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Rows where grain breaks: {len(grain_check)}  (should be 0)")

# Proof 2: row count + date span — already proven in Step 1 (reference that output here)

# Proof 3: availability, filtered with IS TRUE
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM {DAILY}
""").df()
print(avail)

# ---- 3b. Five features ----

features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_month,
        AVG(gsc_avg_position) AS avg_position_month,
        COUNT(DISTINCT report_date) AS days_active_month
    FROM {DAILY}
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
print(features.head())

# ---- 3c. The trap ----

halves = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
           SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM {DAILY}
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

halves['declined_label'] = (halves['clicks_second_half'] < halves['clicks_first_half']).astype(int)
frame = features.merge(halves[['content_hash_id','declined_label','clicks_first_half','clicks_second_half']], on='content_hash_id')

honest_cols = ['impressions_month','clicks_month','ctr_month','avg_position_month','days_active_month']
X_honest = frame[honest_cols].fillna(0)
y = frame['declined_label']
honest_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X_honest, y)
print("Honest score:", honest_tree.score(X_honest, y))

# THE TRAP: add a column literally derived from the label
frame['leaky_pct_change'] = (frame['clicks_second_half'] - frame['clicks_first_half']) / frame['clicks_first_half'].replace(0, np.nan)
X_leaky = frame[honest_cols + ['leaky_pct_change']].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X_leaky, y)
print("Leaky score (watch this jump toward 1.0):", leaky_tree.score(X_leaky, y))

# delete the leak, keep the honest number
frame = frame.drop(columns=['leaky_pct_change'])
print("Final honest score kept:", honest_tree.score(X_honest, y))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where grain breaks: 0  (should be 0)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4  pct_with_ga4
0     9841378       413966.0           4.2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id  impressions_month  \
0  content_b813c73d7000b3b1  client_9958f0a7ae1df715              180.0   
1  content_3a3e193ec1e76e3b  client_9958f0a7ae1df715               89.0   
2  content_5a77dbf5671c5a65  client_9958f0a7ae1df715            19657.0   
3  content_278030b007943b07  client_9958f0a7ae1df715              319.0   
4  content_347fbafb77d3ae37  client_9958f0a7ae1df715              396.0   

   clicks_month  ctr_month  avg_position_month  days_active_month  
0           1.0       0.56            8.674734                  7  
1           1.0       1.12           12.381138                  7  
2         199.0       1.01            4.532382                 31  
3           7.0       2.19            5.914484                 10  
4           4.0       1.01           21.671343                 20  
Honest score: 0.6817182198941307
Leaky score (watch this jump toward 1.0): 1.0
Final honest score kept: 0.6817182198941307


## 4. Data limits
Data limits: only 4.2% of March rows have ga4_data_available IS TRUE. This isn't random noise — it traces back to the client level: of all clients in the warehouse, only some have has_ga4_access = TRUE, and of those, only some had their ga4_data_start before March 2026 (see query above). GA4 is a late-arriving, unevenly-adopted data source. Any feature built from ga4_* columns (like days_active_month filtered on GA4 availability) describes a small, non-random subset of pages — mostly larger, more established clients — not the full March panel. Search-based features (gsc_*) are far more complete and should be trusted more heavily for now. This also means the trap experiment's honest 0.68 score was measured on that same narrow 4.2% slice, not the full month — a caveat worth carrying into modeling weeks.

In [7]:
client_ga4 = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN has_ga4_access THEN 1 ELSE 0 END) AS n_with_ga4_access,
        SUM(CASE WHEN has_ga4_access AND ga4_data_start <= DATE '2026-03-01' THEN 1 ELSE 0 END) AS n_ga4_ready_by_march
    FROM {DIM_CLIENTS}
""").df()
client_ga4


,n_clients,n_with_ga4_access,n_ga4_ready_by_march
0,104,54.0,25.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.